In [0]:
from typing import Callable, Dict, List

from pyspark.sql.functions import col, lit, nullif, to_date, to_timestamp

from pipelines.stream_utils import (
    get_stream_dataframe,
    read_autoloader_stream,
    reset_stream_target,
    write_delta_table_stream,
)

In [0]:
def apply_function_to_columns(columns: List[str], function: callable, args=[]) -> Dict[str, callable]:
    return {column: function(column, *args) for column in columns}

In [0]:
stream_entity = "clicks" 

table_name = "main.silver.clicks"

target_path = "/Volumes/main/silver/ecommerce/clicks"
checkpoint_location = "/Volumes/main/silver/ecommerce/_checkpoints/clicks"

In [0]:
# Reprocess = reset only the target side (table/path/checkpoint), then read all files again.
dbutils.widgets.dropdown("reprocess", "false", ["false", "true"])
reprocess = dbutils.widgets.get("reprocess").lower() == "true"

if reprocess:
    print("reprocess=true: clearing target (table/path/checkpoint) before re-running")
    reset_stream_target(
        table_name=table_name,
        output_path=target_path,
        checkpoint_location=checkpoint_location,
    )

In [0]:
source_stream_df = read_autoloader_stream(
    stream_entity=stream_entity,
    schema_evolution_mode="addNewColumns"
)

In [0]:
silver_stream_df = (
    source_stream_df
    .withColumn("occurred_at", to_timestamp(col("occurred_at")))
    .withColumn("campaign_id", nullif(col("campaign_id"), lit("")))
    .withColumn("ingest_date", to_date(col("ingest_date")))
    .withColumns({
        "is_logged_in": col("is_logged_in").cast("boolean"),
        "component_load_time_ms": col("component_load_time_ms").cast("integer"),
        "page_load_time_ms": col("page_load_time_ms").cast("integer"),
        "session_elapsed_ms": col("session_elapsed_ms").cast("integer"),
        "time_on_previous_page_ms": col("time_on_previous_page_ms").cast("integer"),
    })
    .drop("product_impression_count")
)

In [0]:
silver_query = write_delta_table_stream(
    df=silver_stream_df,
    table_name=table_name,
    checkpoint_location=checkpoint_location,
    query_name="silver_clicks_available_now",
    partition_by=["ingest_date"],
    merge_schema=True
)

silver_query.awaitTermination()